# Exercise 5 - Case study: steam methane reforming vs. water electrolysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/05_case_study_hydrogen.ipynb)

Time to put everything together. Two plants each produce **50 t of hydrogen per day** on the US Gulf Coast:
a **steam methane reforming (SMR)** plant sized from an Aspen Plus simulation, and a pressurized **alkaline
water electrolysis (WE)** plant costed from a US DOE report. You will build both, compare their cost
structures, find the CO₂ tax at which electrolysis becomes competitive, and compare the two under uncertainty.

The data are adapted from case study 1 of the OpenPyTEA repository (Tamarona et al., 2026), where a third
pathway, methane pyrolysis, is also analysed.

This is Exercise 5 of 6. The SMR plant is given: run through it quickly. The
electrolysis plant and the comparison are your tasks.

## Learning goals

- Translate a process flowsheet and a sizing table into an OpenPyTEA equipment list and plant configuration.
- Handle equipment given as vendor quotes, periodic replacement CAPEX, and a manual operator count.
- Compare two technologies with breakdown charts, cash-flow diagrams, and sensitivity curves.
- Find a policy break-even point (the CO₂ tax that equalizes the levelized cost of hydrogen).
- Quantify the probability that one technology is cheaper than the other.

## Background references

- Tamarona, P.B., Vlugt, T.J.H., & Ramdin, M. (2026). OpenPyTEA: An open-source python toolkit for techno-economic assessment of chemical process plants and energy systems with economic sensitivity and uncertainty evaluation. *SoftwareX, 35*, 102816. https://doi.org/10.1016/j.softx.2026.102816
- Acevedo, Y.M., Prosser, J.H., Huya-Kouadio, J.M., McNamara, K.R., & James, B.D. (2023). *Hydrogen Production Cost with Alkaline Electrolysis*. DOE-SA-09629-1, Strategic Analysis Inc. https://doi.org/10.2172/2203367
- Boyano, A., Blanco-Marigorta, A.M., Morosuk, T., & Tsatsaronis, G. (2011). Exergoenvironmental analysis of a steam methane reforming process for hydrogen production. *Energy, 36*(4), 2202-2214. https://doi.org/10.1016/j.energy.2010.05.020

## 0) Setup

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
import numpy as np
import pandas as pd
from copy import deepcopy
from openpytea import Equipment, Plant
from openpytea.analysis import (fixed_capital_data, variable_opex_data, fixed_opex_data, levelized_cost_data,
                                direct_costs_data, cash_flow_data, sensitivity_data, tornado_data, monte_carlo)
from openpytea.plotting import (plot_stacked_bar, plot_cash_flow, plot_sensitivity, plot_tornado,
                                plot_monte_carlo, plot_multiple_monte_carlo)

## 1) Common assumptions

Both plants share the location, financing, and price assumptions below. Prices carry an uncertainty block
straight away so that the Monte Carlo comparison at the end needs no extra configuration.

In [ ]:
country, region   = "United States", "Gulf Coast"
interest_rate     = 0.09
project_lifetime  = 30          # years
plant_utilization = 0.95
tax_rate          = 0.25

h2_production = 50_000          # kg/day (50 t/day)
h2_price      = 5.0             # USD/kg

operator_rate = {"rate": 38.11, "rate_uncertainty": {"std": 10, "min": 25, "max": 50}}          # USD/h

# Prices with their uncertainty: (base, std, min, max)
electricity_price   = {"price": 0.05,      "price_uncertainty": {"std": 0.03,  "min": 0.01,  "max": 3.0}}     # USD/kWh
natural_gas_price   = {"price": 0.40,      "price_uncertainty": {"std": 0.25,  "min": 0.0,   "max": 5.0}}     # USD/kg
cooling_water_price = {"price": 2.4592e-4, "price_uncertainty": {"std": 1e-4,  "min": 1e-5,  "max": 4e-4}}    # USD/MJ
co2_tax             = {"price": 0.075,     "price_uncertainty": {"std": 0.02,  "min": 0.0,   "max": 0.15}}    # USD/kg CO2
water_price         = {"price": 0.003,     "price_uncertainty": {"std": 0.001, "min": 0.0,   "max": 0.005}}   # USD/kg

## 2) Steam methane reforming (given)

Methane is reformed with steam at 700 °C over a nickel catalyst, the syngas is shifted in high- and
low-temperature water-gas-shift reactors, and hydrogen is recovered by pressure-swing adsorption. The tail gas
and extra methane fire the reformer furnace.

![SMR flowsheet](https://raw.githubusercontent.com/pbtamarona/OpenPyTEA/main/examples/img/CS_2-Fig_1.png)

The sizes below come from the Aspen Plus simulation shipped with the OpenPyTEA repository.

In [ ]:
# Compressors and their electric motors (90 % motor efficiency)
comp_1 = Equipment("COMP-1", 945, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")
comp_2 = Equipment("COMP-2", 491, "Fluids", "Compressors, fans, & blowers", "Compressor, centrifugal", "Carbon steel")
mtr_1  = Equipment("MTR-1", 945 / 0.9, "Electrical", "Motors & generators", "Totally enclosed motor", "Carbon steel")
mtr_2  = Equipment("MTR-2", 491 / 0.9, "Electrical", "Motors & generators", "Totally enclosed motor", "Carbon steel")

# Heat exchangers (m2)
hx_1   = Equipment("HX-1",   31.87,  "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
hx_2   = Equipment("HX-2",   22.74,  "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
hx_3   = Equipment("HX-3",   107.19, "Fluids", "Heat Exchangers", "U-tube shell & tube", "304 stainless steel")
cooler = Equipment("COOLER", 191.47, "Fluids", "Heat Exchangers", "U-tube shell & tube", "Carbon steel")

# Reactors (shell mass, kg) and furnace (duty, kW), pump (L/s)
smr_reactor    = Equipment("SMR-REACTOR",    160,     "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel")
wgs_ht_reactor = Equipment("WGS-HT-REACTOR", 376.54,  "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel")
wgs_lt_reactor = Equipment("WGS-LT-REACTOR", 3822.92, "Fluids", "Reactors", "Tubular fixed bed", "304 stainless steel")
furnace = Equipment("FURNACE", 33.26, "Fluids", "Boilers, heaters, & furnaces", "Furnace, cylindrical", "304 stainless steel")
pump    = Equipment("PUMP",    5.16,  "Fluids", "Pumps", "Single-stage centrifugal", "Carbon steel")

# PSA: no verified correlation in the database, so a literature cost (1994 USD) is entered directly
psa = Equipment("PSA", 1023.49, "Fluids", "Adsorption & membrane separation", "Pressure-swing adsorber",
                "Carbon steel", purchased_cost=284_692.37, cost_year=1994)

smr_equipment = [comp_1, mtr_1, comp_2, mtr_2, hx_1, hx_2, hx_3, cooler,
                 smr_reactor, wgs_ht_reactor, wgs_lt_reactor, furnace, pump, psa]

In [ ]:
smr_plant = Plant({
    "plant_name": "Steam reforming",
    "process_type": "Fluids",
    "country": country, "region": region,
    "equipment": smr_equipment,
    "interest_rate": interest_rate, "project_lifetime": project_lifetime,
    "plant_utilization": plant_utilization, "tax_rate": tax_rate,
    "operator_hourly_rate": operator_rate,
    "plant_products": {
        "hydrogen": {"production": h2_production, "price": h2_price},
    },
    "variable_opex_inputs": {                                           # daily quantities from the simulation
        "electricity":   {"consumption": (945 + 491) / 0.9 * 24, **electricity_price},    # kWh/day
        "methane_feed":  {"consumption": (5139 + 4025) * 24,     **natural_gas_price},    # kg/day, feed + fuel
        "cooling_water": {"consumption": 20_222.67 * 24,         **cooling_water_price},  # MJ/day
        "CO2_tax":       {"consumption": 25_124.64 * 24,         **co2_tax},              # kg CO2/day emitted
    },
})
smr_plant.calculate_all()
print(f"SMR  FCI {smr_plant.fixed_capital/1e6:6.1f} MUSD   LCOH {smr_plant.levelized_cost:.2f} USD/kg   NPV {smr_plant.npv/1e6:7.1f} MUSD   IRR {smr_plant.irr:.1%}")

## 3) Task 5.1: build the water-electrolysis plant

The DOE report describes a pressurized alkaline system (about 30 bar) delivering 99.99 % hydrogen without
downstream compression.

![WE flowsheet](https://raw.githubusercontent.com/pbtamarona/OpenPyTEA/main/examples/img/CS_2-Fig_3.png)

Instead of sized units, the report gives **system costs** (2020 USD): stack 17.194 MUSD, mechanical balance of
plant 11.019 MUSD, electrical balance of plant 16.974 MUSD. The system consumes **98 MW** of electricity and
**3.78 kg of water per kg H₂ twice over** (process plus cooling make-up, so 3.78 × 2 × 50 000 kg/day).

**Tasks**
1. Create three `Equipment` items from the quotes (`purchased_cost=..., cost_year=2020`, `param=0`). Use
   `process_type="Electrical"` for the electrical balance of plant and `"Fluids"` for the other two, `category="Electrolysis"`.
2. Create `we_plant` with the common assumptions, `process_type="Fluids"`, hydrogen as product, and electricity
   and water as variable OPEX inputs (reuse `electricity_price` and `water_price`).
3. The automatic operator estimate counts equipment items, which is meaningless for three "systems". The
   flowsheet has 11 fluid process steps: compute `we_plant.calculate_operators_hired(no_fluid_process=11, no_solid_process=0)`
   and set it with `update_configuration({"operators_hired": ...})`.
4. The stack lasts 10 years, so it is replaced in years 10 and 20: set `additional_capex_cost` to the stack's
   (escalated) purchased cost twice and `additional_capex_years` to `[10, 20]`.
5. Run `we_plant.calculate_all(additional_capex=True)` and print FCI, LCOH, NPV, and IRR as for SMR.

In [ ]:
# YOUR CODE HERE

## 4) Comparing cost structures

Pass both plants as a list. For the electrolysis plant, `additional_capex=True` includes the stack replacements
in the capital breakdown.

In [ ]:
plants = [smr_plant, we_plant]
fig, ax = plot_stacked_bar(fixed_capital_data(plants, additional_capex=True))
fig, ax = plot_stacked_bar(variable_opex_data(plants))
fig, ax = plot_stacked_bar(levelized_cost_data(plants))

In [ ]:
fig, ax = plot_stacked_bar(direct_costs_data(smr_plant, pct=True))    # what drives the SMR capital cost?
fig, ax = plot_cash_flow(cash_flow_data(plants))

> Electrolysis is dominated by capital and electricity; reforming by the methane feed and, to a lesser extent,
> the CO₂ tax. At today's assumptions SMR produces hydrogen at about 4 USD/kg and electrolysis at about
> 7.5 USD/kg.

## 5) Sensitivity: what would change the ranking?

In [ ]:
fig, ax = plot_sensitivity(sensitivity_data(plants, parameter="electricity", plus_minus_value=0.6, metric="LCOP",
                                            label="Levelized cost of hydrogen / [USD/kg]", n_points=31))
fig, ax = plot_sensitivity(sensitivity_data(plants, parameter="interest_rate", plus_minus_value=0.6, metric="NPV", n_points=31))

In [ ]:
fig, ax = plot_tornado(tornado_data(we_plant, plus_minus_value=0.3, metric="LCOP", additional_capex=True))
fig, ax = plot_tornado(tornado_data(smr_plant, plus_minus_value=0.3, metric="LCOP"))

## 6) Task 5.2: the CO₂ tax that levels the field

The SMR plant emits about 12 kg CO₂ per kg H₂. At what CO₂ tax (USD per kg CO₂) does the levelized cost of
hydrogen from SMR equal that of electrolysis?

Sweep the tax from 0 to 0.5 USD/kg CO₂ in steps of 0.025: for each value, `deepcopy(smr_plant)`, update
`variable_opex_inputs.CO2_tax.price`, run `calculate_levelized_cost()`, and store the result. Then plot LCOH
versus tax for SMR together with a horizontal line at the electrolysis LCOH, and interpolate the crossing with
`numpy.interp`. Express the answer also in USD per tonne CO₂.

In [ ]:
# YOUR CODE HERE

## 7) Task 5.3: which one is cheaper, and how sure are we?

Every price already carries an uncertainty block, and the default project-level uncertainties are active.

1. Run `monte_carlo` on both plants with 20 000 samples and the same `random_seed` (pass `additional_capex=True` for electrolysis).
2. Plot both LCOH distributions on one figure with `plot_multiple_monte_carlo`.
3. Compute the probability that electrolysis is cheaper than SMR: `np.mean(lcoh_we < lcoh_smr)`.
4. Bonus: what is the probability if the electricity price is a long-term contract at 0.02 USD/kWh with `std` 0.005?

In [ ]:
# YOUR CODE HERE

## 8) Discussion

We pick these questions up together in the wrap-up. If you finish early, discuss them with your neighbour:

- Which three inputs would you spend effort on before showing these results to a decision maker?
- The comparison uses a **class-4 factorial estimate** for SMR and **vendor-level system costs** for electrolysis. Is that a fair comparison? How would you make the accuracy classes consistent?
- What does the CO₂ break-even tax tell a policy maker, and what does it hide (for example, upstream methane emissions, electricity carbon intensity, or learning-curve cost reductions of electrolyzers)?

## Recap

You have built two plants from process data, compared their cost structures, identified their drivers, found a
policy break-even point, and quantified the ranking under uncertainty. That is the complete OpenPyTEA workflow.

**Next:** [Exercise 6 - JSON configuration files and the command line](06_json_and_cli.ipynb)

After the workshop:
- Try the [home assignment](../HOME_ASSIGNMENT.md) on a process of your own.
- Missing a cost correlation? Contribute it: https://github.com/pbtamarona/OpenPyTEA